<p align="center"><img src="../Additionals/Empa-Workshops-Template-Banner.jpg" alt="University Workshops" style="display: block; margin: 0 auto" height=/></p>

# Welcome to the Accelerator Workshops!

Welcome to the Edge AI step of our Accelerator Workshops series organized by Empa Electronics. This open-source repository contains the working environment for the "Developing Edge-AI Solutions for ST Platforms" activity so you can follow the exercises and reproduce the workflow.

This script demonstrates the development steps for an edge AI solution.

**Application Steps:**

1. Include requirements

2. Data preprocessing

3. Create the AI model

4. Train the model (development)

5. Export the model & Deployment

## Installation (Cloud Environment Only)

This section performs automatic setup steps for the cloud environment. Run the cells in the Installation section ONLY in Google Colab. Local environment users should start from the **1. Include Requirements** section.

1. Verify Python 3.10 is available as the preferred Python version.

In [ ]:
!python3 --version

2. Create required folders and download resources.

In [ ]:
!mkdir -p Datasets && mkdir -p Models
!wget https://raw.githubusercontent.com/Empa-Teknoloji/AI_Workshop/master/Activity2_Bare-Metal_Edge-AI_Solution/Datasets/Dataset_Hand_Character_Recognition_EmpaElectronics.csv -O Datasets/Dataset_Hand_Character_Recognition_EmpaElectronics.csv

3. Install the necessary Python packages.

In [ ]:
!pip3 install -r https://raw.githubusercontent.com/Empa-Teknoloji/AI_Workshop/master/Activity2_Bare-Metal_Edge-AI_Solution/requirements.txt

## 1. Include Requirements

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from math import floor
from scipy.stats import mode
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from tensorflow.keras.layers import Dense, Dropout, Conv1D, Flatten, MaxPooling1D
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split

In [ ]:
path_dataset = "./Datasets/Dataset_Hand_Character_Recognition_EmpaElectronics.csv"

## 2. Data Preprocessing

### 2.1. Raw Data Acquisition

In [ ]:
df_dataset = pd.read_csv(path_dataset)
df_dataset

### 2.2. Feature & Labels Separation

In [ ]:
df_feats, df_labels = df_dataset.drop(columns=["class"]), df_dataset["class"]

Feature columns of the dataset:

In [ ]:
df_feats

Label column of the dataset:

In [ ]:
df_labels

Null value checks for feature and label dataframes:

In [ ]:
print(f"[features] Number of NAs: {df_feats.isna().sum().sum()}")
print(f"[features] Number of nulls: {df_feats.isnull().sum().sum()}")
print(f"[labels] Number of NAs: {df_labels.isna().sum().sum()}")
print(f"[labels] Number of nulls: {df_labels.isnull().sum().item()}")

List of classes in the dataset:

In [ ]:
list_categories = np.array(sorted(set(df_labels.to_numpy().flatten()))).reshape(-1, 1)
list_categories

### 2.3. Applying Label Encoding

Call One-Hot Encoder from scikit-learn:

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)

Fit the encoder on dataset classes:

In [ ]:
encoder.fit(list_categories)

List of classes inside the One-Hot Encoder:

In [ ]:
encoder.categories_

Getting one-hot encoded label values:

In [ ]:
df_labels_ohe = pd.DataFrame(encoder.transform(df_labels.to_numpy().reshape(-1, 1)))
df_labels_ohe

### 2.4. Creating Sequence Batches from the Dataset

Define a fixed sequence length and overlap ratio:

Sequence length is the number of samples grouped into a single window rather than evaluated individually.

Example:
For a dataset with 1000 samples and 6 features (1000 x 6), selecting sequence_length = 200 will produce 5 sequence batches of 200 samples each (new dataset shape: 5 x 200 x 6).

In [ ]:
seq_length = 128
overlapping_ratio = 0.33

Define the function that creates sequence batches

In [ ]:
def create_sequences(data, labels, num_samples=104, overlap=0.5):
    """Takes tabular data and creates times sequences with given sample width."""

    # create empty lists for stacking
    data_sequences = []
    data_labels = []
    # get the number of examples
    num_examples = data.shape[0]
    # compute stride value
    strides = num_samples - floor(num_samples * overlap)
    # compute the number of sequences to use as iterator
    num_sequences = floor((num_examples - num_samples) / strides) + 1

    # iterate for sequence range
    for ind in range(num_sequences):

        # define start index
        ind_start = ind * strides
        # define end index
        ind_end = ind_start + num_samples
        # get the current data slice by using start and end indexes
        slice_seq_acc = data.values[ind_start:ind_end]
        # get the current labels slice by using start and end indexes
        slice_seq_label = labels.values[ind_start:ind_end]

        # take the modal value of label slice: (replace with .mode())
        label_seq = mode(slice_seq_label, keepdims=True)[0][0]

        # stack current slices
        data_sequences.append(slice_seq_acc)
        data_labels.append(label_seq)

    # convert stacks to numpy array
    data_sequences = np.array(data_sequences)
    data_labels = np.array(data_labels)

    # return sequence and label stacks as X and Y
    return data_sequences, data_labels

Create the sequence dataset from the original dataset:

In [ ]:
x_feats_seq, y_labels_seq = create_sequences(
                                    data= df_feats,
                                    labels= df_labels_ohe,
                                    num_samples= seq_length,
                                    overlap=overlapping_ratio)

In [ ]:
print("Shape of Sequence Features:" , x_feats_seq.shape)
print("Shape of Sequence Labels:" , y_labels_seq.shape)

### 2.5. Train and Test Dataset Split

Define train/test split ratio:

In [ ]:
test_split_ratio = 0.1

Split the dataset:

In [ ]:
x_train_seq, x_test_seq, y_train_seq, y_test_seq = train_test_split(x_feats_seq, y_labels_seq, test_size=test_split_ratio, shuffle=True)

In [ ]:
print(f"Shapes of Train Set - Features: {x_train_seq.shape} - Labels: {y_train_seq.shape}")
print(f"Shapes of Test Set - Features: {x_test_seq.shape} - Labels: {y_test_seq.shape} ")

## 3. Creating the AI Model

### 3.1. Model Creation

Define model creation function for a 1-D CNN:

In [ ]:
def create_cnn_model(input_shape, output_shape):

        """Creates 1D-CNN model for sequence processing.

        Parameters:
            - input_shape: model input shape
            - output_shape: model output shape (number of classes)
        returns:
            - model: keras Sequential CNN model
        """

        model_cnn = Sequential(name="model_CNN")
        # Layer-1: Conv1D
        model_cnn.add(
            Conv1D(
                filters=64,
                kernel_size=3,
                activation="relu",
                padding="valid",
                strides=1,
                input_shape=input_shape,
            )
        )
        # Layer-2: Conv1D
        model_cnn.add(
            Conv1D(filters=32, kernel_size=3, activation="relu", padding="valid", strides=1)
        )
        # Layer-3: Dropout
        model_cnn.add(Dropout(0.4))
        # Layer-4: MaxPooling
        model_cnn.add(MaxPooling1D(pool_size=2, strides=2))
        # Layer-5: Flattening
        model_cnn.add(Flatten())
        # Layer-6: Fully-Connected
        model_cnn.add(Dense(units=32, activation="relu"))
        # Layer-Output: Softmax
        model_cnn.add(Dense(units=output_shape, activation="softmax"))
        # return CNN model object
        return model_cnn

Next, define the training function that uses the previously defined model creation function:

In [ ]:
def train_model(X, y, max_epochs=500, batch_size=128, lr=0.001, X_val=None, y_val=None):

    # get seq lenght and num of features
    seq_length, num_features = X.shape[1:]
    # get number of features and define input shape
    input_shape_cnn = seq_length, num_features
    # define output shape
    output_classes_cnn = y[0].size
    print("model input shape:", input_shape_cnn)
    print("output input shape:", output_classes_cnn)

    # create CNN model instance
    model = create_cnn_model(input_shape_cnn, output_classes_cnn)

    # compile model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["categorical_accuracy"],
    )
    # define EarlyStopping callback
    callback_early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="loss", min_delta=0, patience=10
    )
    # train a model for current param. combination
    history = model.fit(
        X,
        y,
        epochs=max_epochs,
        batch_size=batch_size,
        callbacks=[callback_early_stopping],
        validation_data=(X_val, y_val), # if  isinstance(None, (type(X_val), type(y_val))) else None,
    )

    # return training history and trained model
    return history, model

### 3.2. Model Training

In [ ]:
history_cnn, model_cnn = train_model(X=x_train_seq,
                                     y=y_train_seq,
                                     X_val=x_test_seq,
                                     y_val=y_test_seq,
                                     lr=0.0003,
)

### 3.3. Evaluating Results

In [ ]:
plt.style.use("ggplot")
plt.plot(history_cnn.history["categorical_accuracy"], label="Train Accuracy")
plt.plot(history_cnn.history["val_categorical_accuracy"], label="Validation Accuracy")
plt.title("Learning Curves")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc="lower right")
plt.show()

Confusion Matrix review:

In [ ]:
# class ID to class name mapping
classes = {0: "CIRCLE",
           1: "HORIZONTAL",
           2: "STANDBY",
           3: "TRIANGLE",
           4: "VERTICAL"}

# plotting the confusion matrix
plt.style.use("default")
y_pred = model_cnn(x_test_seq)
y_pred_argmaxed = np.argmax(y_pred, axis=1)
y_test_argmaxed = np.argmax(y_test_seq.astype(int), axis=1)
y_pred_named = np.vectorize(classes.get)(y_pred_argmaxed)
y_test_named = np.vectorize(classes.get)(y_test_argmaxed)
ConfusionMatrixDisplay.from_predictions(y_test_named, y_pred_named)
plt.xticks(rotation=90)
plt.show()

## 4. Exporting the Trained Model

To convert the trained model (architecture + learned parameters) to a deployable artifact, save it to a static file.

Get the model accuracy from the training history:

In [ ]:
accuracy = history_cnn.history["val_categorical_accuracy"][-1]
accuracy = round(accuracy, 4)
accuracy_for_naming = str(accuracy).replace(".", "_")
accuracy

### 4.1. Export: H5 File and/or Keras

In [ ]:
model_cnn.save(f"Models/Model_CNN_Hand_Character_Recognition_{accuracy_for_naming}.h5")

In [ ]:
model_cnn.save(f"Models/Model_CNN_Hand_Character_Recognition_{accuracy_for_naming}.keras")

### 4.2. Export: TFLite

In [ ]:
# Convert the model.
converter = tf.lite.TFLiteConverter.from_keras_model(model_cnn)
tflite_model = converter.convert()

# Save the model.
with open(f'Models/Model_CNN_Hand_Character_Recognition_{accuracy_for_naming}.tflite', 'wb') as f:
  f.write(tflite_model)

### 4.3. Export: ONNX

In [ ]:
import tf2onnx
import onnx
import onnxruntime as ort

In [ ]:
x_val = np.ones((1, 128, 6), np.float32)
input_signature = [tf.TensorSpec([None, 128, 6], tf.float32, name='x')]
onnx_model, _ = tf2onnx.convert.from_keras(model_cnn, input_signature, opset=13)

print("Keras result")
print(model_cnn(x_val).numpy())

print("ONNX RunTime result")
sess = ort.InferenceSession(onnx_model.SerializeToString())
res = sess.run(None, {'x': x_val})
print(res[0])

In [ ]:
onnx.save(onnx_model, f"Models/Model_CNN_Hand_Character_Recognition_{accuracy_for_naming}.onnx")

## 5. Testing the Saved Model

In [ ]:
import tensorflow as tf
model_loaded = tf.keras.models.load_model(f"Models/Model_CNN_Hand_Character_Recognition_{accuracy_for_naming}.keras")

In [ ]:
model_loaded.summary()

## 6. Edge Deployment using the Trained Model

_To deploy the trained model on ST platforms, continue with the STM32Cube.AI tool._